# Calculating Possesion per Shot

Calculating how many possessions a team needs on average to generate a shot revealing insight into how effective teams are with their possession.

---

**Change Directory**

This block of code is only ran so that, so this notebook can use the FMA library.

In [ ]:
from pathlib import Path
import sys
import os

cwd = os.getcwd()
os.chdir(cwd)

project_root = Path.cwd().parent
sys.path.append(str(project_root))

**Import FMA Library**

In [ ]:
from FootballMatchAnalysis.objects.match import Match

**Load Event and Tracking Data**

In [ ]:
DATADIR = './../data'
game_id = 1


In [ ]:
match = Match(DATADIR, game_id)

**Time On Ball**

`time_on_ball` returns a list containing all of the individual player possessions (time a player is "on ball") in a match. This data was generated while `match = Match(DATADIR, game_id)` was ran and is determined by iterating over every event and determining who is on the ball and when.

In [ ]:
possesions = match.time_on_ball

A possession is represented as a dictionary containing: 

- Who is on the ball
- What team are the player on
- What frame did they receive the ball
- What frame did their possession ended

In [ ]:
possesions[0]

**Calculating Number of Possessions**

For the sake of reuse, we will functionalize this code. This code is counting the number of possessions a team has during the match. 

A possession begins when the team in focus receives the ball and ends once the opposing team receives the ball. While individual `possession` dictionaries from `time_on_ball` contain one player, possessions, for the sake of this calculation, can span multiple players of the same team.

In [ ]:
def number_of_possessions(possesions, team):
    num_possesions = 0
    in_possesion = True

    # Iterate over each possesssion
    for possesion in possesions:
        # Is the team aleady in possession?
        if in_possesion:
            # Is the other team currently in possession?
            if possesion["Team"] != team:
                # End the possession
                in_possesion = False
        else:
            # Is the team currently in possession?
            if possesion["Team"] == team:
                # Start a new possession
                num_possesions += 1
                in_possesion = True
    
    return num_possesions

**Calculate Number of Shots**


For the sake of reuse, we will functionalize this code, but we are attempting to count the number of shots a team has during the match.

In [ ]:
def number_of_shots(team):
    # Get all shots events
    shots = match.get_events("SHOT")

    # Filter to only shots by given team
    shots = shots[shots["Team"] == team]

    # Count and return number of remaining shots
    num_shots = len(shots)
    return num_shots

**Calculating Possesion per Shot**

Using the functions we've written, we will use them to calculate the possessions per shot of each team over the course of the match.

In [ ]:
team = 'Home'

home_num_shots = number_of_shots(team)
home_num_possesions = number_of_possessions(possesions, team)
home_possesions_per_shot = round(home_num_possesions/home_num_shots, 2)

In [ ]:
team = 'Away'

away_num_shots = number_of_shots(team)
away_num_possesions = number_of_possessions(possesions, team)
away_possesions_per_shot = round(away_num_possesions/away_num_shots, 2)

In [ ]:
print("Home Team")
print(f"  Shots: {home_num_shots}")
print(f"  Possessions: {home_num_possesions}")
print(f"  Possessions per Shot: {home_possesions_per_shot}")
print("Away Team")
print(f"  Shots: {away_num_shots}")
print(f"  Possessions: {away_num_possesions}")
print(f"  Possessions per Shot: {away_possesions_per_shot}")

Reviewing the results, both the home and away teams had the same number of possessions. This result makes sense considering that when a team loses possession of the ball, the other team receives it . This result could be altered, if you wanted, by setting a minimum time threshold for a given team's possessions during `number_of_possessions`.

*Example: The Home team has the ball. The away team deflects a pass for a frame, but lands at the feet of the home. Curretnly this is 2 Home team possessions and 1 Away team possession.*

The home team outshot the away team 18-6. The difference is visible when you see the gulf between both teams' possessions per shot. In the case of the Away team, they needed to get on the ball +26 times just to get a single shot off. The home team was much more efficient with their possessions.